# Advanced EDA & Feature Engineering — UNSW-NB15**Project 1 | DecodeLabs Industrial Training | Kinza Arshad****Goal:** Transform the raw UNSW-NB15 network traffic dataset into a mathematically clean, production-ready feature set.**Pipeline (Input → Process → Output):**1. Structural inspection2. Missing-value decision matrix3. IQR-based outlier neutralization (winsorization, not deletion)4. Categorical encoding (coordinate-space translation)5. Domain-specific feature engineering (5 new features)6. Multicollinearity eradication7. Pandera runtime data contract8. Clean output export**Dataset:** UNSW-NB15 (binary train/test set), a labeled network intrusion detection dataset — 9 attack categories + normal traffic, ~49 features covering flow, TCP, content, and time-based statistics.

## 0. Setup & Configuration

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statspd.set_option('display.max_columns', 100)sns.set_theme(style='whitegrid', palette='deep')plt.rcParams['figure.dpi'] = 110RANDOM_STATE = 42TRAIN_PATH = 'UNSW_NB15_training-set.parquet'TEST_PATH  = 'UNSW_NB15_testing-set.parquet'

## 1. Load & Structural InspectionWe load with `pd.read_parquet()` (binary format — requires `pyarrow` or `fastparquet`, install with `pip install pyarrow` if needed).**What to expect:** ~175,341 training rows, ~82,332 testing rows, 45 feature columns + `attack_cat` (multi-class label) + `label` (binary: 0=normal, 1=attack).

In [ ]:
df = pd.read_parquet(TRAIN_PATH)df_test = pd.read_parquet(TEST_PATH)print('Train shape:', df.shape)print('Test shape :', df_test.shape)print('\nMemory usage (train):', round(df.memory_usage(deep=True).sum() / 1024**2, 2), 'MB')df.info()

In [ ]:
df.head()

### 1.1 Target DistributionCheck class balance for both `label` (binary) and `attack_cat` (multi-class). UNSW-NB15 is known to be **imbalanced across attack categories** — this matters later for how you'll train models, and it also tells you whether some categorical/numeric features need group-aware imputation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2E86AB', '#E63946'])axes[0].set_title('Binary Label Distribution (0=Normal, 1=Attack)')axes[0].set_xticklabels(['Normal', 'Attack'], rotation=0)df['attack_cat'].value_counts().plot(kind='barh', ax=axes[1], color='#457B9D')axes[1].set_title('Attack Category Distribution')axes[1].invert_yaxis()plt.tight_layout()plt.show()print(df['attack_cat'].value_counts(normalize=True).round(3) * 100)

## 2. Missing Value Decision MatrixPer the training brief: don't guess — apply structural thresholds.| Missingness | Action ||---|---|| < 5% | Row deletion (`dropna`) — preserves distribution, negligible data loss || 5–20% | Statistical imputation — median for skewed numeric, sub-group conditional for categorical/correlated || > 20% | KNN imputation — captures multi-dimensional relationships |UNSW-NB15 is usually clean of true NaNs (it's a curated benchmark dataset), but it **does encode missing/unknown categorical values as `'-'`** in `service` and sometimes placeholder values in numeric fields — these need to be treated as missingness, not taken at face value.

In [ ]:
# Step 1: Convert known placeholder codes to actual NaN so pandas can see themdf['service'] = df['service'].replace('-', np.nan)missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)missing_pct = missing_pct[missing_pct > 0]print('Columns with missing values:')print(missing_pct)

In [ ]:
def apply_missing_value_matrix(data, col):    """Applies the <5% / 5-20% / >20% decision matrix to a single column."""    pct = data[col].isnull().mean() * 100    if pct == 0:        return data, 'no action needed'    if pct < 5:        data = data.dropna(subset=[col])        action = f'row deletion ({pct:.2f}% missing)'    elif pct <= 20:        if data[col].dtype == 'object':            # sub-group conditional imputation: mode within attack_cat group            data[col] = data.groupby('attack_cat')[col].transform(                lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else 'unknown')            )            action = f'sub-group conditional mode imputation ({pct:.2f}% missing)'        else:            data[col] = data[col].fillna(data[col].median())            action = f'global median imputation ({pct:.2f}% missing)'    else:        from sklearn.impute import KNNImputer        imputer = KNNImputer(n_neighbors=5)        numeric_subset = data.select_dtypes(include=[np.number])        imputed = imputer.fit_transform(numeric_subset)        data[numeric_subset.columns] = imputed        action = f'KNN imputation, k=5 ({pct:.2f}% missing)'    return data, actionfor col in missing_pct.index:    df, action_taken = apply_missing_value_matrix(df, col)    print(f'{col}: {action_taken}')print('\nShape after missing-value treatment:', df.shape)print('Remaining nulls:', df.isnull().sum().sum())

## 3. Outlier Detection & Neutralization (IQR + Winsorization)**Why winsorization, not deletion:** this dataset has ~175K rows and time-ordered network flow records. Dropping rows destroys sequential integrity and shrinks minority attack classes further. `numpy.clip()` caps values at the statistical boundary instead — preserves row count and distribution shape.Bounds: `Lower = Q1 - 1.5*IQR`, `Upper = Q3 + 1.5*IQR`

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()numeric_cols = [c for c in numeric_cols if c != 'label']  # never touch the targetoutlier_summary = []def iqr_winsorize(data, col):    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)    IQR = Q3 - Q1    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR    n_outliers = ((data[col] < lower) | (data[col] > upper)).sum()    data[col] = data[col].clip(lower=lower, upper=upper)    return data, n_outliers, lower, upperfor col in numeric_cols:    df, n_out, lo, hi = iqr_winsorize(df, col)    if n_out > 0:        outlier_summary.append({'feature': col, 'outliers_capped': n_out,                                 'lower_bound': round(lo, 3), 'upper_bound': round(hi, 3)})outlier_df = pd.DataFrame(outlier_summary).sort_values('outliers_capped', ascending=False)outlier_df.head(15)

In [ ]:
# Visualize before/after effect on the most-affected featuretop_feature = outlier_df.iloc[0]['feature']fig, ax = plt.subplots(1, 1, figsize=(8, 4))sns.boxplot(x=df[top_feature], ax=ax, color='#2E86AB')ax.set_title(f'{top_feature} distribution AFTER winsorization')plt.tight_layout()plt.show()print(f'Total outlier values capped across {len(outlier_df)} features: {outlier_df["outliers_capped"].sum()}')

## 4. Categorical Encoding — Coordinate-Space Translation**Rule:** Label Encoding introduces a false mathematical hierarchy (Tokyo ≠ 3× London). We only use it where a natural order exists. Otherwise:- **Low cardinality (< 10 unique values)** → One-Hot Encoding (`proto` has ~3 relevant top values + 'other', `state` ~8 values)- **High cardinality (`service`, `proto` full range)** → Frequency Encoding, to avoid exploding dimensionality with One-Hot

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()cat_cols = [c for c in cat_cols if c != 'attack_cat']  # keep attack_cat as reference label, not a featureprint('Categorical columns:', cat_cols)for c in cat_cols:    print(f'  {c}: {df[c].nunique()} unique values')

In [ ]:
# proto and service tend to be high-cardinality with a long tail -> frequency encoding# state is low-cardinality -> one-hot encodingHIGH_CARD_THRESHOLD = 10for col in cat_cols:    n_unique = df[col].nunique()    if n_unique <= HIGH_CARD_THRESHOLD:        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)        df = pd.concat([df, dummies], axis=1)        df.drop(columns=[col], inplace=True)        print(f'{col}: one-hot encoded -> {dummies.shape[1]} new columns')    else:        freq_map = df[col].value_counts(normalize=True)        df[col + '_freq'] = df[col].map(freq_map)        df.drop(columns=[col], inplace=True)        print(f'{col}: frequency encoded -> 1 new column ({col}_freq)')print('\nShape after encoding:', df.shape)

## 5. Domain-Specific Feature EngineeringGeneric features (row sums, simple ratios of arbitrary columns) don't demonstrate domain understanding. These 5 features are built from **network-traffic domain logic** — the kind of features a security analyst would actually reason about:1. **`byte_ratio`** — src-to-dst byte ratio. Attacks like DoS often show extreme asymmetry (huge outbound, near-zero inbound).2. **`pkt_size_avg_src`** — average packet size from source (`sbytes / spkts`). Scans tend to send many tiny packets.3. **`total_pkt_rate`** — total packets per second of flow duration. Exfiltration/flood attacks push abnormal packet rates.4. **`tcp_setup_ratio`** — TCP handshake timing ratio (`tcprtt / (synack + ackdat + 1e-6)`). Reconnaissance/spoofing shows abnormal TCP setup timing.5. **`conn_load_ratio`** — source-to-destination load ratio (`sload / (dload + 1e-6)`). Distinguishes push-heavy (attack) vs. balanced (normal) sessions.All ratios use `+ 1e-6` in denominators to avoid division-by-zero without introducing NaN/inf, which would break the pipeline downstream.

In [ ]:
EPS = 1e-6df['byte_ratio']       = df['sbytes'] / (df['dbytes'] + EPS)df['pkt_size_avg_src']  = df['sbytes'] / (df['spkts'] + EPS)df['total_pkt_rate']    = (df['spkts'] + df['dpkts']) / (df['dur'] + EPS)df['tcp_setup_ratio']   = df['tcprtt'] / (df['synack'] + df['ackdat'] + EPS)df['conn_load_ratio']   = df['sload'] / (df['dload'] + EPS)new_features = ['byte_ratio', 'pkt_size_avg_src', 'total_pkt_rate', 'tcp_setup_ratio', 'conn_load_ratio']# Cap the new ratio features too -- ratios are especially prone to extreme valuesfor col in new_features:    df, n_out, lo, hi = iqr_winsorize(df, col)    print(f'{col}: {n_out} extreme ratio values capped')df[new_features].describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))for ax, feat in zip(axes.flatten(), new_features):    sns.boxplot(x=df['label'], y=df[feat], ax=ax, palette=['#2E86AB', '#E63946'])    ax.set_title(feat)    ax.set_xticklabels(['Normal', 'Attack'])axes.flatten()[-1].axis('off')plt.tight_layout()plt.show()

## 6. Multicollinearity EradicationHigh correlation between predictors makes `X.T @ X` singular/unstable — coefficients become impossible to trust. We:1. Build the absolute correlation matrix2. Isolate the upper triangle (avoid double-counting pairs)3. Flag pairs with correlation > 0.854. For each flagged pair, keep whichever feature correlates more strongly with `label`, drop the other

In [ ]:
corr_matrix = df[numeric_cols + new_features].corr().abs()upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))high_corr_pairs = [(col, row, upper.loc[row, col])                    for col in upper.columns for row in upper.index                    if pd.notnull(upper.loc[row, col]) and upper.loc[row, col] > 0.85]print(f'Found {len(high_corr_pairs)} highly correlated pairs (>0.85):')target_corr = df[numeric_cols + new_features].corrwith(df['label']).abs()to_drop = set()for feat_a, feat_b, corr_val in high_corr_pairs:    weaker = feat_a if target_corr.get(feat_a, 0) < target_corr.get(feat_b, 0) else feat_b    to_drop.add(weaker)    print(f'  {feat_a} <-> {feat_b} (r={corr_val:.2f})  -->  dropping {weaker}')df.drop(columns=list(to_drop), inplace=True, errors='ignore')print(f'\nDropped {len(to_drop)} redundant features. New shape: {df.shape}')

In [ ]:
plt.figure(figsize=(14, 10))remaining_numeric = df.select_dtypes(include=[np.number]).columns.tolist()sns.heatmap(df[remaining_numeric].corr(), cmap='coolwarm', center=0, linewidths=0.3)plt.title('Correlation Matrix — After Multicollinearity Eradication')plt.tight_layout()plt.show()

## 7. Runtime Data Contract (Pandera)Treat this cleaned dataframe as a **software interface**, not a one-off script output. This schema is what a production pipeline would validate against before serving data to a model — it catches silent corruption (nulls sneaking back in, dtype drift, out-of-range values) before it reaches training or inference.Install if needed: `pip install pandera`

In [ ]:
import pandera as pafrom pandera import Column, Check, DataFrameSchemaschema = DataFrameSchema({    'dur':   Column(float, Check.ge(0), nullable=False),    'sbytes': Column(float, Check.ge(0), nullable=False),    'dbytes': Column(float, Check.ge(0), nullable=False),    'byte_ratio': Column(float, nullable=False),    'label': Column(int, Check.isin([0, 1]), nullable=False),}, strict=False)  # strict=False: only validate the columns we listed, ignore the resttry:    schema.validate(df, lazy=True)    print('Schema validation PASSED — data contract satisfied.')except pa.errors.SchemaErrors as err:    print('Schema validation FAILED. Failure cases:')    print(err.failure_cases)

## 8. Export Clean DatasetSave the processed, validated dataframe as the output of this pipeline stage. In a real feature-store setup (Feast), this would be the offline store write — the same logic would be reused for online serving to avoid training-serving skew.

In [ ]:
output_path = 'UNSW_NB15_train_processed.parquet'df.to_parquet(output_path, index=False)print(f'Saved: {output_path}')print(f'Final shape: {df.shape}')print(f'Final columns ({len(df.columns)}):')print(list(df.columns))

## Summary| Stage | What was done | Rows affected / Result ||---|---|---|| Missing values | Threshold-based deletion / imputation / KNN | See Section 2 output || Outliers | IQR winsorization (capped, not dropped) | See Section 3 output || Encoding | One-hot (low-card) + frequency (high-card) | `proto`, `service`, `state` transformed || Feature engineering | 5 domain-specific ratio features | `byte_ratio`, `pkt_size_avg_src`, `total_pkt_rate`, `tcp_setup_ratio`, `conn_load_ratio` || Multicollinearity | Correlation-based, target-aware pruning | See Section 6 output || Data contract | Pandera schema, lazy validation | Pass/fail printed above |### Next steps (not in this notebook)- Repeat this exact pipeline on `UNSW_NB15_testing-set.parquet` using the **same fitted encoders/bounds** from train (don't refit on test — that's leakage)- Baseline model (Logistic Regression / XGBoost) on `label`, then multi-class on `attack_cat`- Wrap this as a reusable `sklearn.Pipeline` / `ColumnTransformer` for production reuse- If you want to push this toward Data Engineering: turn this notebook into a scripted ETL job (`prefect`/`airflow` DAG) that runs on new data drops